# add_damage_record.ipynb
Add a new damages record to the database.

In [ ]:
%matplotlib widget
import ipympl
import matplotlib.pyplot as plt
import sqlite3
from icecream import ic 
import numpy as np
import shapely
from shapely import wkt
from pyefd import elliptic_fourier_descriptors, calculate_dc_coefficients, reconstruct_contour
import cv2
from shapely.geometry import Polygon
from shapely.wkt import dumps
import re
from contextlib import contextmanager

In [ ]:
db_path = 'Efate2025B_4k.db'
# image_id = 1
# tree_id = 1
order = 15

In [ ]:
@contextmanager
def open_db(db_path=db_path):
    conn = sqlite3.connect(db_path)
    try:
        # 1. Configure Row Factory (built-in dictionary-like rows)
        conn.row_factory = sqlite3.Row
        
        # 2. Load SpatiaLite Extension
        conn.enable_load_extension(True)
        conn.load_extension("mod_spatialite") 
        
        # Yield the fully configured connection
        yield conn
        
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

# Usage example:

# db_path = 'new.db'
# with open_db() as conn:
#     cursor = conn.cursor()
    
#     # You can now use SpatiaLite functions right away
#     cursor.execute("SELECT image_id FROM images WHERE damage_flag = 0")
    
#     # Because of sqlite3.Row, you can access columns by name!
#     image_id_queue = [row['image_id'] for row in cursor.fetchall()]    
    
#     # Connection is automatically committed and closed here.
    
# ic(image_id_queue);       

In [ ]:
!jupyter notebook --version

In [ ]:
def calc_damage_polygons(image_height, image_width, original_contour, reconstructed_contour):
    """  
    returns mask containing pixels which are in the reconstruction mask but not in the original mask
    """
    original_mask = np.zeros((image_height, image_width), dtype=np.uint8)
    cv2.fillPoly(original_mask, pts=[original_contour.astype(dtype=np.int32)], color=255)

    reconstructed_mask = np.zeros((image_height, image_width), dtype=np.uint8)
    cv2.fillPoly(reconstructed_mask, pts=[reconstructed_contour.astype(dtype=np.int32)], color=255)

    # Pixels missing in original, present in reconstruction
    diff_mask = cv2.bitwise_and(reconstructed_mask, cv2.bitwise_not(original_mask))

    # remove small blobs using connected components method
    #######################################################

    # Load your binary image (must be single-channel, uint8)
    # binary_img = cv2.cvtColor(diff_mask, cv2.COLOR_BGR2GRAY)
    # ic(binary_img.shape)
    binary_img = diff_mask

    # 1. Find all connected components and their statistics
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_img, connectivity=8)

    # 2. Define your minimum area threshold (in pixels)
    min_area = 200 

    # 3. Create an empty mask to store large objects
    filtered_img = np.zeros_like(binary_img)

    # 4. Loop through components (skip index 0 as it represents the background)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        
        if area >= min_area:
            # Keep components that meet the size constraint
            filtered_img[labels == i] = 255
            

    # Find connected components
    num_labels, labels_im = cv2.connectedComponents(filtered_img)

    # The background is counted as a label, so subtract 1 to get the blob count
    blob_count = num_labels - 1

    # ic(blob_count)
    ###########################################################################3        

    # ic(diff_mask)
    # ic(np.max(diff_mask))

    # gray_mask = cv2.cvtColor(diff_mask, cv2.COLOR_BGR2GRAY)
    damage_polygons, _ = cv2.findContours(
        image=filtered_img, 
        mode=cv2.RETR_EXTERNAL, 
        method=cv2.CHAIN_APPROX_NONE)
    return damage_polygons

# # Usage example:
# damage_polygons = calc_damage_polygons(original_contour, reconstructed_contour)
# ic(len(damage_polygons));
# ic(damage_polygons[0])


# fig, ax = plt.subplots()
# ax.imshow(filtered_img)
# plt.show()

In [ ]:
def add_damage_polygons_to_db(image_id, tree_id, image_polygons):
    """  
    Add damage polygons to database
    """

    def wkt_to_ints(wkt_str):
        # Find all floating point or integer numbers in the WKT string
        return re.sub(r'-?\d+\.\d+', lambda m: str(round(float(m.group(0)))), wkt_str)

    damage_wkts = []
    for poly in damage_polygons:
        # poly = poly.astype(int) 
        poly = np.squeeze(poly)         # get rid of unnecessary dimensions
        if (poly[-1] != poly[0]).all(): # ensure polygon is closed
            np.vstack([poly, poly[0]])
        damage_wkt = dumps(Polygon(poly))
        damage_wkt = wkt_to_ints(damage_wkt)          # convert coords from float to int
        damage_wkts.append(damage_wkt)    

    with open_db() as conn:
        cursor = conn.cursor()
        for damage_wkt in damage_wkts:   # add records to damage table
            cursor.execute('INSERT INTO damage (image_id, tree_id, damage_poly) VALUES (?, ?, GeomFromText(?, 0))', 
                            (image_id, tree_id, damage_wkt))

In [ ]:
# MAIN

# open database

# add damage_flag field to images table if it does not exist THIS BLOCK IS TEMPORARY
with open_db() as conn: 
    cursor = conn.cursor()
    try:
        # Attempt to add the column
        cursor.execute("ALTER TABLE images ADD COLUMN damage_flag INTEGER NOT NULL DEFAULT 0;")
        conn.commit()
        ic("damage_flag column added successfully!")
    except sqlite3.OperationalError as e:
        # If the error is because the column already exists, ignore it safely
        if "duplicate column name" in str(e):
            ic("damage_flag column already exists. Skipping.")
        else:
            # Re-raise if it's a completely different database error
            raise e

# clear all values from damage table and set damage_flag to 0; TEMP FOR TESTING
with open_db() as conn:
    cursor = conn.cursor()
    cursor.execute('DELETE from damage')
    ic('all values cleared from damage table for testing')
    cursor.execute('UPDATE images SET damage_flag = 0')
    ic('damage_flag set to 0 for all records in images table for testing')

while True: 
           
    # get image_id for next image requiring damage shape processing
    with open_db() as conn: 
        cursor = conn.cursor()       
        cursor.execute("SELECT image_id, image_height, image_width FROM images WHERE damage_flag = 0 LIMIT 1")
        row = cursor.fetchone()
        
    if row:
        image_id = row['image_id']
        ic(image_id)
        image_height = row['image_height']
        image_width = row['image_width']
    else:
        ic('there are 0 images requiring damage shape processing. EXITING MAIN') 
        break

    # get tree_id_queue containing a list of tree_ids for damage shape processing 
    with open_db() as conn:
        cursor = conn.cursor()
        cursor.execute(f'SELECT tree_id FROM trees WHERE image_id={image_id}')
        tree_id_queue = [row['tree_id'] for row in cursor.fetchall()]
        
    ic(len(tree_id_queue))  
    if len(tree_id_queue) > 0:
        for tree_id in tree_id_queue: # process tree_queue        

            # get WKT for tree polygon
            with open_db() as conn: 
                cursor = conn.cursor()       
                cursor.execute(f'SELECT AsText(tree_poly) AS tree_wkt FROM trees WHERE tree_id={tree_id}')
                tree_wkt = cursor.fetchone()['tree_wkt']
                # ic(tree_id, tree_wkt)
                
            # get coords
                
            # 1. Load the WKT into a Shapely geometry object
            polygon = wkt.loads(tree_wkt)

            # 2. Extract the exterior coordinates and convert to a NumPy array
            coords_array = np.array(polygon.exterior.coords)
            coords_array = np.abs(coords_array)  # Ensure all coordinates are positive
            
            # get elliptic Fourier descriptors

            original_contour = coords_array
            original_contour = np.array(original_contour, dtype=np.float64).reshape(-1, 2)
            # ic(original_contour)

            # 1. Extract descriptors (keep normalize=False to retain native spatial details)
            coeffs = elliptic_fourier_descriptors(original_contour, order=order, normalize=False)
            # ic(coeffs)

            # 2. Extract DC spatial tracking offsets
            dc_coeffs = calculate_dc_coefficients(original_contour)
            # ic(dc_coeffs)

            # 3. Generate raw reconstruction (matching original row length for point parity)
            reconstructed_contour = reconstruct_contour(
                coeffs, 
                locus=dc_coeffs, 
                num_points=len(original_contour)
            )
            # ic(reconstructed_contour)
            
            damage_polygons = calc_damage_polygons(image_height, image_width, original_contour, reconstructed_contour)
            add_damage_polygons_to_db(image_id, tree_id, damage_polygons) 
            
    # set damage_flag to 1  to mark damage shape processing is finished for current image_id
    with open_db() as conn: 
        cursor = conn.cursor()
        cursor.execute(f'UPDATE images SET damage_flag = 1 WHERE image_id = {image_id}')      
            

In [ ]:
ic('FINISHED')